In [1]:
import requests
import csv
import json

In [2]:
baseURL = "https://www.loc.gov/"
endpoint = "free-to-use/"
collection = "cats/"
parameters = {"fo": "json"}

In [3]:
collection_list = requests.get(baseURL + endpoint + collection, params=parameters)
with open("ftu-cats-set-info.json", "w", encoding="utf-8") as f:
        f.write(json.dumps(collection_list.json(), indent=2))

In [4]:
headers = ["alt", "image", "link", "title"]

with open("ftu-cats-set-list.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader()
    for item in collection_list.json()["content"]["components"][0]["masonry_gallery"][
        0
    ]["items"]:
        item["title"] = item["title"].rstrip()
        writer.writerow(item)
print(
    f"Wrote {len(collection_list.json()['content']['components'][0]['masonry_gallery'][0]['items'])} rows to ftu-cats-set-list.csv"
)

Wrote 40 rows to ftu-cats-set-list.csv


In [2]:
from pathlib import Path
import glob
import os
from os.path import join

In [6]:
def regenerate_collection_list(collection_csv):
    coll_items = []
    with open(collection_csv, "r", newline="", encoding="utf-8") as f:
        data = csv.DictReader(f)
        for row in data:
            row_dict = dict(row)
            for field in data.fieldnames:
                row_dict[field] = row[field]
            coll_items.append(row_dict)
        return coll_items

In [7]:
collection_csv = Path("ftu-cats-set-list.csv")
collection_set_list = regenerate_collection_list(collection_csv)

In [8]:
baseURL = "https://www.loc.gov"
parameters = {"fo": "json"}

In [9]:
item_metadata_directory = Path("item-metadata").mkdir(exist_ok=True)

In [10]:
item_count = 0
error_count = 0
file_count = 0

item_metadata_file_prefix = "item-metadata"
json_suffix = ".json"

for item in collection_set_list:
    if item["link"] == "link":
        continue
    if "?" in item["link"]:
        resource_ID = item["link"]
        short_ID = item["link"].split("/")[2]
        item_metadata = requests.get(baseURL + resource_ID, params=parameters)
        print("Requested", item_metadata.url, item_metadata.status_code)
        if item_metadata.status_code != 200:
            print("Requested", item_metadata.url, item_metadata.status_code)
            error_count += 1
            continue
        try:
            item_metadata.json()
        except:
            error_count += 1
            print("No json found")
            continue
        fout = os.path.join(
            "item-metadata", item_metadata_file_prefix + "-" + short_ID + json_suffix
        )
        with open(fout, "w", encoding="utf-8") as f:
            f.write(json.dumps(item_metadata.json()["item"], indent=2))
            file_count += 1
            print("Wrote", fout)
        item_count += 1
    else:
        resource_ID = item["link"]
        short_ID = item["link"].split("/")[2]
        item_metadata = requests.get(baseURL + resource_ID, params=parameters)
        print("Requested", item_metadata.url, item_metadata.status_code)
        if item_metadata.status_code != 200:
            print("Requested", item_metadata.url, item_metadata.status_code)
            error_count += 1
            continue
        try:
            item_metadata.json()
        except:
            error_count += 1
            print("No json found")
            continue
        fout = os.path.join(
            "item-metadata", item_metadata_file_prefix + "-" + short_ID + json_suffix
        )
        with open(fout, "w", encoding="utf-8") as f:
            f.write(json.dumps(item_metadata.json()["item"], indent=2))
            file_count += 1
            print("Wrote", fout)
        item_count += 1

print("Items requested:", item_count)
print("Errors:", error_count)
print("Files written:", file_count)

Requested https://www.loc.gov/resource/ppmsca.15528/?fo=json 200
Wrote item-metadata/item-metadata-ppmsca.15528.json
Requested https://www.loc.gov/resource/pga.12042/?fo=json 200
Wrote item-metadata/item-metadata-pga.12042.json
Requested https://www.loc.gov/resource/cph.3c21044/?fo=json 200
Wrote item-metadata/item-metadata-cph.3c21044.json
Requested https://www.loc.gov/resource/jpd.02425/?fo=json 200
Wrote item-metadata/item-metadata-jpd.02425.json
Requested https://www.loc.gov/resource/jpd.02798/?fo=json 200
Wrote item-metadata/item-metadata-jpd.02798.json
Requested https://www.loc.gov/resource/cph.3b06249/?fo=json 200
Wrote item-metadata/item-metadata-cph.3b06249.json
Requested https://www.loc.gov/resource/cph.3g08753/?fo=json 200
Wrote item-metadata/item-metadata-cph.3g08753.json
Requested https://www.loc.gov/resource/pga.05046/?fo=json 200
Wrote item-metadata/item-metadata-pga.05046.json
Requested https://www.loc.gov/resource/pga.07149/?fo=json 200
Wrote item-metadata/item-metadat

In [3]:
files_directory = Path("item-files").mkdir(exist_ok=True)

In [4]:
search_for_metadata = Path("item-metadata")
metadata_file_list = list(search_for_metadata.glob("*.json"))

In [13]:
item_image_urls = []
count = 0

for item in metadata_file_list:
    with open(item, "r", encoding="utf-8") as f:
        metadata = json.load(f)
        image_url_no = len(metadata["image_url"])
        image_url = metadata["image_url"][-1]
        item_image_urls.append(image_url)
        count += 1

print(f"Identified {count} image URLS")

Identified 40 image URLS


In [14]:
collection_set_list_with_images = []

for item in metadata_file_list:
    with open(item, "r", encoding="utf-8") as f:
        item_metadata = json.load(f)
        item_metadata_dict = dict(item_metadata)
        item_metadata_dict["item_URI"] = item_metadata["id"]
        try:
            item_metadata_dict["lccn"] = item_metadata[
                "library_of_congress_control_number"
            ]
        except:
            item_metadata_dict["lccn"] = None
        item_metadata_dict["title"] = item_metadata["title"]
        item_metadata_dict["image_URL_large"] = item_metadata["image_url"][-1]
        collection_set_list_with_images.append(item_metadata_dict)

In [15]:
item_count = 0
error_count = 0
file_count = 0

img_file_prefix = "item"
img_file_suffix = "image.jpg"

for item in collection_set_list_with_images:
    image_URL = item["image_URL_large"]
    short_ID = item["item_URI"].split("/")[4]
    print("Requested", image_URL)
    item_count += 1
    r = requests.get(image_URL)
    if r.status_code == 200:
        img_out = os.path.join(
            "item-files", img_file_prefix + "-" + short_ID + "-" + img_file_suffix
        )
        with open(img_out, "wb") as f:
            f.write(r.content)
            print("Wrote", img_out)
            file_count += 1

print("Files requested:", item_count)
print("Errors:", error_count)
print("Files written:", file_count)

Requested https://tile.loc.gov/storage-services/service/pnp/jpd/02700/02798v.jpg#h=1024&w=750
Wrote item-files/item-2009631922-image.jpg
Requested https://tile.loc.gov/storage-services/service/pnp/cph/3g00000/3g03000/3g03000/3g03063v.jpg#h=1024&w=817
Wrote item-files/item-94508810-image.jpg
Requested https://tile.loc.gov/storage-services/service/pnp/cph/3c20000/3c21000/3c21000/3c21044v.jpg#h=823&w=1024
Wrote item-files/item-98514094-image.jpg
Requested https://tile.loc.gov/image-services/iiif/service:pnp:highsm:43900:43955/full/pct:25/0/default.jpg#h=1228&w=1840
Wrote item-files/item-2017656550-image.jpg
Requested https://tile.loc.gov/storage-services/service/pnp/jpd/02400/02425v.jpg#h=813&w=1024
Wrote item-files/item-2008660988-image.jpg
Requested https://tile.loc.gov/image-services/iiif/public:rbc:2003juv0001:0001/full/pct:100.0/0/default.jpg#h=4627&w=4294
Wrote item-files/item-03028132-image.jpg
Requested https://tile.loc.gov/storage-services/service/pnp/npcc/10800/10856v.jpg#h=1024

In [16]:
current_loc = os.getcwd()

print(current_loc)

/Users/fallonmccue/si676/cats-collection


In [4]:
metadata_file_path = Path("item-metadata")

print(metadata_file_path)

item-metadata


In [5]:
for file in metadata_file_path.glob("item-metadata-*.json"):
    print(file)

item-metadata/item-metadata-jpd.02798.json
item-metadata/item-metadata-cph.3g03063.json
item-metadata/item-metadata-cph.3c21044.json
item-metadata/item-metadata-highsm.43955.json
item-metadata/item-metadata-jpd.02425.json
item-metadata/item-metadata-rbc0001.2003juv0001.json
item-metadata/item-metadata-npcc.10856.json
item-metadata/item-metadata-ppmsca.51533.json
item-metadata/item-metadata-cph.3c20459.json
item-metadata/item-metadata-ppmsca.15528.json
item-metadata/item-metadata-cph.3b16490.json
item-metadata/item-metadata-ds.04034.json
item-metadata/item-metadata-hec.21589.json
item-metadata/item-metadata-highsm.41489.json
item-metadata/item-metadata-cai.2a15065.json
item-metadata/item-metadata-pga.12042.json
item-metadata/item-metadata-pga.05046.json
item-metadata/item-metadata-pga.07149.json
item-metadata/item-metadata-cph.3g03065.json
item-metadata/item-metadata-ppmsca.05919.json
item-metadata/item-metadata-gsc.5a10626.json
item-metadata/item-metadata-cph.3b39357.json
item-metadata

In [3]:
list_of_item_metadata_files = list()
for file in glob.glob("item-metadata/item-metadata-*.json"):
    list_of_item_metadata_files.append(file)

In [4]:
len(list_of_item_metadata_files)

40

In [5]:
list_of_item_metadata_files.sort()

for file in list_of_item_metadata_files:
    print(file)

item-metadata/item-metadata-agc.7a16795.json
item-metadata/item-metadata-agc.7a17761.json
item-metadata/item-metadata-cai.2a15065.json
item-metadata/item-metadata-cph.3b06249.json
item-metadata/item-metadata-cph.3b16490.json
item-metadata/item-metadata-cph.3b39357.json
item-metadata/item-metadata-cph.3b52525.json
item-metadata/item-metadata-cph.3c00170.json
item-metadata/item-metadata-cph.3c06978.json
item-metadata/item-metadata-cph.3c20459.json
item-metadata/item-metadata-cph.3c21044.json
item-metadata/item-metadata-cph.3g03063.json
item-metadata/item-metadata-cph.3g03065.json
item-metadata/item-metadata-cph.3g08217.json
item-metadata/item-metadata-cph.3g08753.json
item-metadata/item-metadata-ds.04034.json
item-metadata/item-metadata-ds.04037.json
item-metadata/item-metadata-ds.07446.json
item-metadata/item-metadata-fsa.8c01765.json
item-metadata/item-metadata-gottlieb.10861.0.json
item-metadata/item-metadata-gsc.5a10626.json
item-metadata/item-metadata-hec.21589.json
item-metadata/it

In [6]:
from datetime import date

date_string_for_today = date.today().strftime("%Y-%m-%d")

In [9]:
collection_info_csv = "collection_items_data.csv"
file_count = 0
items_written = 0
error_count = 0

headers = [
    "creator",
    "contributor",
    "date",
    "date_uploaded",
    "description",
    "extent",
    "format",
    "identifier",
    "item_type",
    "language",
    "location",
    "physical_format",
    "rights",
    "source",
    "subject",
    "title",
    "type",
    "image_url",
]

for item_file in list_of_item_metadata_files:
    file_count += 1
    print("Opening", item_file)
    with open(item_file, "r", encoding="utf-8") as data:
        try:
            item_data = json.load(data)
        except:
            print("Error loading", item_file)
            error_count += 1
            continue
        try:
            item_creator = item_data["contributor_names"][0]
        except:
            item_creator = None
        try:
            for contributor in item_data["contributor_names"]:
                if contributor == item_creator:
                    item_contributor = None
                else:
                    item_contributor = contributor
        except:
            item_contributor = None
        try:
            item_date = item_data["date"]
        except:
            item_date = None
        try:
            item_date_uploaded = date_string_for_today
        except:
            item_date_uploaded = None
        try:
            item_description = " ".join(item_data["notes"])
        except:
            item_description = None
        try:
            item_extent = item_data["medium"][0]
        except:
            item_extent = None
        try:
            item_format = item_data["mime_type"][1].split("/")[1].upper()
        except:
            item_format = None
        item_identifier = "nc_" + str(file_count)
        item = "Image"
        try:
            item_language = ", ".join(item_data["language"])
        except:
            item_language = None
        try:
            item_location = ", ".join(item_data["location"])
        except:
            item_location = None
        try:
            item_physical_format = ", ".join(item_data["original_format"])
        except:
            item_physical_format = None
        try:
            item_rights = item_data["rights_information"]
        except:
            item_rights = None
        try:
            item_source = item_data["id"]
        except:
            item_source = None
        try:
            item_subject = ", ".join(item_data["subject"])
        except:
            item_subject = None
        try:
            item_title = item_data["title"]
        except:
            item_title = None
        try:
            item_type = item_data["item"]["format"][0]
        except:
            item_type = None
        try:
            item_image_url = item_data["image_url"][0]
        except:
            item_image_url = None
    row_dict = dict()
    row_dict["creator"] = item_creator
    row_dict["contributor"] = item_contributor
    row_dict["date"] = item_date
    row_dict["date_uploaded"] = item_date_uploaded
    row_dict["description"] = item_description
    row_dict["extent"] = item_extent
    row_dict["format"] = item_format
    row_dict["identifier"] = item_identifier
    row_dict["item_type"] = item
    row_dict["language"] = item_language
    row_dict["location"] = item_location
    row_dict["physical_format"] = item_physical_format
    row_dict["rights"] = item_rights
    row_dict["source"] = item_source
    row_dict["subject"] = item_subject
    row_dict["title"] = item_title
    row_dict["type"] = item_type
    row_dict["image_url"] = item_image_url
    with open(collection_info_csv, "a", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        if file_count == 1:
            writer.writeheader()
        writer.writerow(row_dict)
        items_written += 1
        print("Adding", item_file)

print("Wrote", collection_info_csv)
print("With", items_written, "items")
print(error_count, "errors (info not written)")

Opening item-metadata/item-metadata-agc.7a16795.json
Adding item-metadata/item-metadata-agc.7a16795.json
Opening item-metadata/item-metadata-agc.7a17761.json
Adding item-metadata/item-metadata-agc.7a17761.json
Opening item-metadata/item-metadata-cai.2a15065.json
Adding item-metadata/item-metadata-cai.2a15065.json
Opening item-metadata/item-metadata-cph.3b06249.json
Adding item-metadata/item-metadata-cph.3b06249.json
Opening item-metadata/item-metadata-cph.3b16490.json
Adding item-metadata/item-metadata-cph.3b16490.json
Opening item-metadata/item-metadata-cph.3b39357.json
Adding item-metadata/item-metadata-cph.3b39357.json
Opening item-metadata/item-metadata-cph.3b52525.json
Adding item-metadata/item-metadata-cph.3b52525.json
Opening item-metadata/item-metadata-cph.3c00170.json
Adding item-metadata/item-metadata-cph.3c00170.json
Opening item-metadata/item-metadata-cph.3c06978.json
Adding item-metadata/item-metadata-cph.3c06978.json
Opening item-metadata/item-metadata-cph.3c20459.json
Ad